# 08 — Ensemble and Weighting

Тук гледаме model weights, reliability scores и weighted ensemble outputs.


In [ ]:
from pathlib import Path
from itertools import combinations
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"


def read_csv_safe(path, **kwargs):
    path = Path(path)
    if not path.exists():
        print(f"Missing file: {path}")
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)


def number_columns(df: pd.DataFrame) -> list[str]:
    return [col for col in ["n1", "n2", "n3", "n4", "n5", "n6"] if col in df.columns]


def parse_numbers_text(value) -> list[int]:
    if pd.isna(value):
        return []
    parts = str(value).replace(";", ",").replace("|", ",").split(",")
    nums = []
    for part in parts:
        part = part.strip()
        if part.isdigit():
            nums.append(int(part))
    return nums


def extract_ticket_numbers(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    if len(number_columns(df)) == 6:
        return df.copy()
    text_col = next((c for c in df.columns if "numbers" in c.lower() or "числа" in c.lower()), None)
    if text_col is None:
        return df.copy()
    out = df.copy()
    parsed = out[text_col].apply(parse_numbers_text)
    for i in range(6):
        out[f"n{i+1}"] = parsed.apply(lambda nums: nums[i] if len(nums) > i else np.nan)
    return out


def show_latest_draw(df: pd.DataFrame) -> None:
    if df.empty:
        print("No draw data loaded.")
        return
    latest = df.tail(1).iloc[0]
    nums = [int(latest[col]) for col in number_columns(df)]
    print("Rows:", len(df))
    print("Latest date:", latest.get("date", "n/a"))
    print("Draw number:", latest.get("draw_number", latest.get("draw_no", "n/a")))
    print("Numbers:", nums)


def file_status(paths):
    rows = []
    for path in paths:
        path = Path(path)
        rows.append({
            "file": str(path.relative_to(PROJECT_ROOT)) if str(path).startswith(str(PROJECT_ROOT)) else str(path),
            "exists": path.exists(),
            "size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else 0,
        })
    return pd.DataFrame(rows)


draws = read_csv_safe(DATA_DIR / "historical_draws.csv")
normalized = read_csv_safe(DATA_DIR / "v40_normalized_draw_events.csv")
canonical = read_csv_safe(DATA_DIR / "v41_canonical_draw_events.csv")
show_latest_draw(draws)


## Model weights


In [ ]:
weights = read_csv_safe(REPORTS_DIR / "v65_model_weights.csv")
weights


In [ ]:
if not weights.empty:
    numeric_cols = weights.select_dtypes(include="number").columns.tolist()
    label_col = next((c for c in weights.columns if "model" in c.lower() or "label" in c.lower()), weights.columns[0])
    preferred = [c for c in ["adaptive_weight_percent", "reliability_score", "adjusted_score"] if c in weights.columns]
    for metric in preferred or numeric_cols[:3]:
        ax = weights.set_index(label_col)[metric].plot(kind="bar", figsize=(11, 4), title=f"Model weighting: {metric}")
        ax.set_xlabel("Model")
        ax.set_ylabel(metric)
        plt.show()
else:
    print("No model weights file found.")


## Weighted ensemble scores


In [ ]:
ensemble = read_csv_safe(REPORTS_DIR / "v66_weighted_smart_ensemble_scores.csv")
display(ensemble.head(20))
if not ensemble.empty and "weighted_score" in ensemble.columns:
    top = ensemble.sort_values("weighted_score", ascending=False).head(15)
    ax = top.set_index("number")["weighted_score"].plot(kind="bar", figsize=(10, 4), title="Top weighted ensemble numbers")
    ax.set_xlabel("Number")
    ax.set_ylabel("weighted_score")
    plt.show()
